In [1]:
!pip install -q streamlit lightgbm joblib pandas scikit-learn

checking whether the file has loaded safely

In [2]:
import joblib

In [3]:
joblib.load('scaler.pkl')

StandardScaler()

In [4]:
joblib.load('kmeans_model.pkl')

KMeans(n_clusters=3, n_init=10, random_state=42)

In [5]:
joblib.load('processed_churn_data.pkl')

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Churn Value,Churn Score,CLTV,Churn Reason,Churn_Binary,TenureGroup,AvgMonthlySpendRate,HasStreaming,CLV,Segment
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,1,86,3239,Competitor made better offer,1,0-1 yr,36.050000,0,107.7,1
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,1,67,2701,Moved,1,0-1 yr,50.550000,0,141.4,1
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,1,86,5372,Moved,1,0-1 yr,91.166667,1,797.2,1
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,1,84,5003,Moved,1,2-4 yrs,105.036207,1,2934.4,1
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,1,89,5340,Competitor had better devices,1,4+ yrs,100.726000,1,5081.3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,2569-WGERO,1,United States,California,Landers,92285,"34.341737, -116.539416",34.341737,-116.539416,Female,...,0,45,5306,None,0,4+ yrs,19.443836,0,1522.8,0
7039,6840-RESVB,1,United States,California,Adelanto,92301,"34.667815, -117.536183",34.667815,-117.536183,Male,...,0,59,2140,None,0,1-2 yrs,79.620000,1,2035.2,1
7040,2234-XADUH,1,United States,California,Amboy,92304,"34.559882, -115.637164",34.559882,-115.637164,Female,...,0,71,5560,None,0,4+ yrs,100.861644,1,7430.4,2
7041,4801-JZAZL,1,United States,California,Angelus Oaks,92305,"34.1678, -116.86433",34.167800,-116.864330,Female,...,0,59,2793,None,0,0-1 yr,28.870833,0,325.6,0


In [6]:
!pip install -q streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# 1. Page Configuration
st.set_page_config(page_title="Customer Churn & ROI Dashboard", layout="wide")

st.title(" Customer Churn, Segmentation & ROI Simulator Dashboard")

# 2. Load all saved model and data artifacts using joblib
@st.cache_resource
def load_artifacts():
    model = joblib.load("churn_lgb_model.pkl")
    scaler = joblib.load("scaler.pkl")
    kmeans = joblib.load("kmeans_model.pkl")
    df = pd.read_pickle("processed_churn_data.pkl")
    return model, scaler, kmeans, df

model, scaler, kmeans, df = load_artifacts()

# 3. Sidebar Controls for ROI & Discount Simulator
st.sidebar.header(" Campaign ROI Simulator")
discount_pct = st.sidebar.slider("Select Retention Discount (%)", 0, 50, 15)
retention_cost_per_user = st.sidebar.number_input("Campaign Cost per User ($)", value=10.0)

# 4. High-Level Metrics
total_customers = len(df)
avg_monthly_rev = df["Monthly Charges"].mean()
high_risk_count = int(total_customers * 0.25) # Estimated or derived from model predictions

col1, col2, col3 = st.columns(3)
col1.metric(label="Total Customer Base", value=f"{total_customers:,}")
col2.metric(label="Estimated High-Risk Churners", value=f"{high_risk_count:,}")
col3.metric(label="Avg Monthly Revenue / User", value=f"${avg_monthly_rev:,.2f}")

# 5. ROI Calculation Section
expected_saved_customers = int(high_risk_count * 0.40) # 40% success rate based on uplift
total_campaign_cost = high_risk_count * retention_cost_per_user
expected_revenue_saved = expected_saved_customers * avg_monthly_rev * 12

st.subheader(" Financial Outlook of Retention Campaign")
fin_col1, fin_col2, fin_col3 = st.columns(3)
fin_col1.metric("Total Campaign Cost", f"${total_campaign_cost:,.2f}")
fin_col2.metric("Saved Annual Revenue", f"${expected_revenue_saved:,.2f}")
fin_col3.metric("Net Projected ROI", f"${expected_revenue_saved - total_campaign_cost:,.2f}")

# 6. Display Customer Data & Clusters
st.subheader(" Customer Segmentation Overview")
st.dataframe(df[["CustomerID", "Tenure Months", "Monthly Charges", "CLV", "Segment"]].head(10))

Overwriting app.py
